# OpcodeShield v1.1.0 — Google Colab Edition
Scan, AI-audit (with our custom trained **`OpcodeShield-SecNet-v1`** GPU model), sanitize, and integrity-verify Windows (`.exe`, `.dll`) and Android (`.apk`) binaries directly in Google Colab.

- **Auto-Update Built In:** Automatically pulls the latest OpcodeShield release on startup.
- **Fast Auto-Download Custom Model:** Automatically downloads `OpcodeShield-SecNet-v1` (~28 MB FP16 safetensors) from GitHub Releases Fastly CDN.

In [ ]:
#@title 1. Install & Auto-Update OpcodeShield + Custom AI Model
import os, subprocess
REPO_DIR = '/content/OpcodeShield'
if os.path.isdir(REPO_DIR):
    print('[AUTO-UPDATE] Updating OpcodeShield to latest main...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/FlaxyDitto-dev/OpcodeShield.git', REPO_DIR], check=True)

subprocess.run(['pip', 'install', '-q', 'pefile', 'psutil', 'safetensors'], check=True)
os.chdir(REPO_DIR)

from custom_model import OpcodeShieldSecNetClient
ai = OpcodeShieldSecNetClient(use_cuda=True, log_callback=print)
ai.ensure_loaded()
print('OpcodeShield v1.1.0 + OpcodeShield-SecNet-v1 ready!')

In [ ]:
#@title 2. Upload a Binary (.exe, .dll, or .apk) to Scan, AI-Audit & Sanitize
from google.colab import files
from pathlib import Path
import subprocess

uploaded = files.upload()
for fname in uploaded.keys():
    p = Path('/content/OpcodeShield') / fname
    print(f'\n=== Static Scan: {fname} ===')
    subprocess.run(['python3', 'cli.py', 'scan', str(p)], check=False)
    print(f'\n=== Custom AI Audit (OpcodeShield-SecNet-v1): {fname} ===')
    subprocess.run(['python3', 'cli.py', 'ai-scan', str(p)], check=False)
    out_p = p.with_name(f'{p.stem}_sanitized{p.suffix}')
    print(f'\n=== Sanitize & Integrity Verify: {fname} -> {out_p.name} ===')
    subprocess.run(['python3', 'cli.py', 'sanitize', str(p), '-o', str(out_p)], check=False)
    if out_p.is_file():
        files.download(str(out_p))